# HapticCodec 2000-sample Colab Workflow

This notebook restores the prepared dataset from Google Drive, builds a fixed 2000-sample subset, trains the HapticCodec reconstruction stage, trains the control branch, and produces reconstruction/control outputs.

In [ ]:
# Experiment configuration
REPO_URL = "https://github.com/cindy-77jiayi/thesis_hapticAE.git"
BRANCH = "codex/wavcaps-vae-cleanup"
REPO_DIR = "/content/thesis_hapticAE"

RUN_NAME = "hapticcodec_subset_2000"
OUTPUT_ROOT = "/content/outputs"
OUTPUT_RUN_DIR = f"{OUTPUT_ROOT}/{RUN_NAME}"

PREPARED_ARCHIVE_DRIVE = "/content/drive/MyDrive/thesis_wavcaps_as/prepared_dataset/wavcaps_haptic_prepared_as.tar"
PREPARED_RESTORE_ROOT = "/content/data"
PREPARED_DATA_DIR = f"{PREPARED_RESTORE_ROOT}/wavcaps_haptic_prepared_as"
SUBSET_ROOT = f"/content/data/subsets/{RUN_NAME}"
SUBSET_DATA_DIR = f"{SUBSET_ROOT}/data"
TRAIN_LIST_PATH = f"{SUBSET_ROOT}/train_list.txt"
VAL_LIST_PATH = f"{SUBSET_ROOT}/val_list.txt"
SUBSET_MANIFEST_PATH = f"{SUBSET_ROOT}/subset_manifest.jsonl"
RUNTIME_CONFIG_DIR = "/content/runtime_configs"
CODEC_CONFIG_PATH = f"{RUNTIME_CONFIG_DIR}/{RUN_NAME}_codec.yaml"
CONTROL_CONFIG_PATH = f"{RUNTIME_CONFIG_DIR}/{RUN_NAME}_control.yaml"
EXPORT_DIR = f"/content/drive/MyDrive/thesis_wavcaps_as/experiments/{RUN_NAME}"

SUBSET_SIZE = 2000
SUBSET_SEED = 42
TRAIN_SPLIT = 0.8

SR = 8000
T = 40000
BATCH_SIZE = 8
ANALYSIS_BATCH_SIZE = 8
CODEC_EPOCHS = 50
CONTROL_EPOCHS = 40

CHANNELS = [32, 64, 128, 128, 64]
STRIDES = [5, 4, 4, 2, 2]
FIRST_KERNEL = 25
KERNEL_SIZE = 9
RESIDUAL_KERNEL_SIZE = 7
CODE_DIM = 64
N_CODEBOOKS = 4
CODEBOOK_SIZE = 256
COMMITMENT_WEIGHT = 0.25
CONTROL_DIM = 16
CONTROL_HIDDEN = 128

LR = 2e-4
WEIGHT_DECAY = 1e-5
L1_WEIGHT = 0.4
RECON_TIME_WEIGHT = 1.0
STFT_LOSS_WEIGHT = 0.5
AMPLITUDE_WEIGHT = 1.0
ENVELOPE_WEIGHT = 1.0
STFT_LINEAR_WEIGHT = 0.02
STFT_LOG_WEIGHT = 0.02

CONTROL_WAVEFORM_L1_WEIGHT = 0.3
CONTROL_AMPLITUDE_WEIGHT = 1.0
CONTROL_ENVELOPE_WEIGHT = 1.0
CONTROL_METRIC_WEIGHT = 1.0
CONTROL_LR = 2e-4

CONTROL_METRICS = [
    "rms_energy",
    "spectral_centroid_hz",
    "spectral_flatness",
    "attack_time_s",
    "gap_ratio",
    "am_modulation_index",
    "short_term_variance",
    "envelope_entropy_bits",
]

In [ ]:
# Clone repo and install deps
!git clone "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git checkout "$BRANCH"
!pip install -q -r requirements.txt pyyaml soundfile librosa scikit-learn tqdm matplotlib

In [ ]:
# Mount Google Drive and restore the prepared dataset archive
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount("/content/drive")
Path(PREPARED_RESTORE_ROOT).mkdir(parents=True, exist_ok=True)
assert Path(PREPARED_ARCHIVE_DRIVE).exists(), f"Missing archive: {PREPARED_ARCHIVE_DRIVE}"
if not Path(PREPARED_DATA_DIR).exists():
    subprocess.run(["tar", "-xf", PREPARED_ARCHIVE_DRIVE, "-C", PREPARED_RESTORE_ROOT], check=True)
print("Prepared dataset:", PREPARED_DATA_DIR)
!find "$PREPARED_DATA_DIR" -name "*.wav" | wc -l

In [ ]:
# Build reproducible 2000-sample subset using symlinks
from pathlib import Path
import json, os, random

prepared_root = Path(PREPARED_DATA_DIR)
subset_root = Path(SUBSET_ROOT)
subset_data = Path(SUBSET_DATA_DIR)
subset_root.mkdir(parents=True, exist_ok=True)
subset_data.mkdir(parents=True, exist_ok=True)

wav_files = sorted(prepared_root.rglob("*.wav"))
assert len(wav_files) >= SUBSET_SIZE, f"Need at least {SUBSET_SIZE} wavs, found {len(wav_files)}"

rng = random.Random(SUBSET_SEED)
selected = sorted(rng.sample(wav_files, SUBSET_SIZE))
train_count = int(SUBSET_SIZE * TRAIN_SPLIT)
train_files = selected[:train_count]
val_files = selected[train_count:]
train_set = set(train_files)

for src in selected:
    rel = src.relative_to(prepared_root)
    dst = subset_data / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    os.symlink(src, dst)

    meta_src = src.with_suffix(".am1.json")
    if meta_src.exists():
        meta_dst = dst.with_suffix(".am1.json")
        if meta_dst.exists() or meta_dst.is_symlink():
            meta_dst.unlink()
        os.symlink(meta_src, meta_dst)

with open(TRAIN_LIST_PATH, "w", encoding="utf-8") as fp:
    for path in train_files:
        fp.write(str(path) + "\n")
with open(VAL_LIST_PATH, "w", encoding="utf-8") as fp:
    for path in val_files:
        fp.write(str(path) + "\n")
with open(SUBSET_MANIFEST_PATH, "w", encoding="utf-8") as fp:
    for path in selected:
        row = {
            "wav": str(path),
            "split": "train" if path in train_set else "val",
        }
        fp.write(json.dumps(row) + "\n")

print("Subset built:", subset_root)
print("train:", len(train_files), "val:", len(val_files))


In [ ]:
# Quick sanity check on the subset
import json, random
from pathlib import Path
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt

sample_wavs = random.sample(sorted(Path(SUBSET_DATA_DIR).rglob("*.wav")), 4)
fig, axes = plt.subplots(len(sample_wavs), 1, figsize=(14, 2.5 * len(sample_wavs)))
if len(sample_wavs) == 1:
    axes = [axes]
for ax, wav_path in zip(axes, sample_wavs):
    y, sr = sf.read(wav_path)
    ax.plot(y, linewidth=0.7)
    ax.set_title(wav_path.name)
    meta_path = wav_path.with_suffix(".am1.json")
    if meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        print(wav_path.name, "->", meta.get("user_prompt", ""))
plt.tight_layout()
plt.show()

In [ ]:
# Write runtime YAML configs so hyperparameters live in the notebook, not the repo
from pathlib import Path
import yaml

Path(RUNTIME_CONFIG_DIR).mkdir(parents=True, exist_ok=True)

base_cfg = {
    "seed": SUBSET_SEED,
    "model_type": "codec",
    "data": {
        "sr": SR,
        "T": T,
        "scale": 1.0,
        "segment_mode": "hapticgen",
        "normalize_mode": "none",
        "min_segment_ratio": 1.0,
        "clip_range": None,
        "train_split": TRAIN_SPLIT,
        "analysis_batch_size": ANALYSIS_BATCH_SIZE,
        "train_random_seek": True,
        "train_sample_with_replacement": False,
        "val_random_seek": False,
        "val_sample_with_replacement": False,
        "train_file_list": TRAIN_LIST_PATH,
        "val_file_list": VAL_LIST_PATH,
    },
    "model": {
        "channels": CHANNELS,
        "strides": STRIDES,
        "first_kernel": FIRST_KERNEL,
        "kernel_size": KERNEL_SIZE,
        "residual_kernel_size": RESIDUAL_KERNEL_SIZE,
        "activation": "leaky_relu",
        "norm": "group",
        "code_dim": CODE_DIM,
        "n_codebooks": N_CODEBOOKS,
        "codebook_size": CODEBOOK_SIZE,
        "commitment_weight": COMMITMENT_WEIGHT,
        "control_dim": CONTROL_DIM,
        "control_hidden": CONTROL_HIDDEN,
        "metric_dim": len(CONTROL_METRICS),
    },
    "training": {
        "batch_size": BATCH_SIZE,
        "epochs": CODEC_EPOCHS,
        "patience": 10,
        "min_delta": 1e-4,
        "early_stop_start": 8,
        "grad_clip": 1.0,
        "print_every": 5,
    },
    "optimizer": {
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
    },
    "scheduler": {
        "factor": 0.5,
        "patience": 8,
    },
    "loss": {
        "l1_weight": L1_WEIGHT,
        "stft_loss_weight": STFT_LOSS_WEIGHT,
        "amplitude_weight": AMPLITUDE_WEIGHT,
        "envelope_weight": ENVELOPE_WEIGHT,
        "clamp_range": 3.0,
        "recon_time_weight": RECON_TIME_WEIGHT,
        "stft_scales": [128, 256, 512, 1024],
        "stft_hop_lengths": [32, 64, 128, 256],
        "stft_win_lengths": [128, 256, 512, 1024],
        "stft_scale_weights": [1.0, 1.0, 0.8, 0.6],
        "stft_linear_weight": STFT_LINEAR_WEIGHT,
        "stft_log_weight": STFT_LOG_WEIGHT,
        "stft_eps": 1e-7,
    },
    "control_training": {
        "batch_size": BATCH_SIZE,
        "epochs": CONTROL_EPOCHS,
        "patience": 8,
        "min_delta": 1e-4,
        "early_stop_start": 6,
        "grad_clip": 1.0,
        "print_every": 5,
    },
    "control_optimizer": {
        "lr": CONTROL_LR,
        "weight_decay": WEIGHT_DECAY,
    },
    "control_scheduler": {
        "factor": 0.5,
        "patience": 6,
    },
    "control_loss": {
        "waveform_l1_weight": CONTROL_WAVEFORM_L1_WEIGHT,
        "amplitude_weight": CONTROL_AMPLITUDE_WEIGHT,
        "envelope_weight": CONTROL_ENVELOPE_WEIGHT,
        "metric_weight": CONTROL_METRIC_WEIGHT,
        "clamp_range": 3.0,
        "metric_names": CONTROL_METRICS,
    },
}

with open(CODEC_CONFIG_PATH, "w", encoding="utf-8") as fp:
    yaml.safe_dump(base_cfg, fp, sort_keys=False)
with open(CONTROL_CONFIG_PATH, "w", encoding="utf-8") as fp:
    yaml.safe_dump(base_cfg, fp, sort_keys=False)

print(CODEC_CONFIG_PATH)
print(CONTROL_CONFIG_PATH)

In [ ]:
# Train codec
%cd "$REPO_DIR"
!python scripts/train_codec.py --config "$CODEC_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --output_root "$OUTPUT_ROOT" --run_name "$RUN_NAME"

In [ ]:
# Reconstruction evaluation
CODEC_CKPT = f"{OUTPUT_RUN_DIR}/codec/best_model.pt"
RECON_EVAL_DIR = f"{OUTPUT_RUN_DIR}/reconstruction_eval"
!python scripts/eval_reconstruction.py --config "$CODEC_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --checkpoint "$CODEC_CKPT" --output_dir "$RECON_EVAL_DIR" --n_samples 10

In [ ]:
# Train control branch
!python scripts/train_control.py --config "$CONTROL_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --codec_checkpoint "$CODEC_CKPT" --output_root "$OUTPUT_ROOT" --run_name "$RUN_NAME"

In [ ]:
# Extract control latents and fit PCA
CONTROL_CKPT = f"{OUTPUT_RUN_DIR}/control/best_control.pt"
PCA_DIR = f"{OUTPUT_RUN_DIR}/pca"
!python scripts/extract_controls.py --config "$CONTROL_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --checkpoint "$CONTROL_CKPT" --output_dir "$PCA_DIR"

In [ ]:
# Build controls spec / sweep gallery / table
CONTROLS_DIR = f"{OUTPUT_RUN_DIR}/controls"
!python scripts/build_controls.py --config "$CONTROL_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --checkpoint "$CONTROL_CKPT" --output_dir "$CONTROLS_DIR" --pca_dir "$PCA_DIR"

In [ ]:
# Extended validation
VALIDATION_DIR = f"{OUTPUT_RUN_DIR}/validation"
!python scripts/validate_extended.py --config "$CONTROL_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --checkpoint "$CONTROL_CKPT" --output_dir "$VALIDATION_DIR" --controls_dir "$CONTROLS_DIR" --pca_dir "$PCA_DIR"

In [ ]:
# Save outputs and runtime metadata back to Drive
from pathlib import Path
import json, shutil

export_root = Path(EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)
for src in [OUTPUT_RUN_DIR, CODEC_CONFIG_PATH, CONTROL_CONFIG_PATH, TRAIN_LIST_PATH, VAL_LIST_PATH, SUBSET_MANIFEST_PATH]:
    src_path = Path(src)
    dst_path = export_root / src_path.name
    if src_path.is_dir():
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

summary = {
    "run_name": RUN_NAME,
    "codec_checkpoint": CODEC_CKPT,
    "control_checkpoint": CONTROL_CKPT,
    "prepared_data_dir": PREPARED_DATA_DIR,
    "subset_root": SUBSET_ROOT,
    "export_dir": str(export_root),
}
with open(export_root / "run_summary.json", "w", encoding="utf-8") as fp:
    json.dump(summary, fp, indent=2)
print("Saved artifacts to:", export_root)